Гипотеза: с внедрением кнопки "С этим товаром покупают" конверсия в покупку увеличится на 2%

In [1]:
import pandas as pd
import clickhouse_connect
import numpy as np

In [5]:
connection = clickhouse_connect.get_client(host='localhost', port=8123, username='etl_user', password='')

query = '''
SELECT
    user_id,
    min(toDate(event_time)) AS event_date,
    max(if(event_type='purchase',1,0)) AS is_purchased,
    sum(if(event_type = 'purchase', price, 0)) as total_revenue,
    avgIf(price, event_type = 'purchase') as avg_check,
    max(price) as max_viewed_price,
    count() as events_count,    
    argMax(brand, price) as favorite_brand
    
FROM electronics_events

WHERE user_id NOT IN (    
    SELECT user_id
    FROM electronics_events
    GROUP BY user_id
    HAVING COUNT() > 10000
) AND toDate(event_time) BETWEEN '2019-10-7' AND '2019-10-21'

GROUP BY user_id
'''

In [6]:
df = connection.query_df(query)

In [7]:
df.head(4)

,user_id,event_date,is_purchased,total_revenue,avg_check,max_viewed_price,events_count,favorite_brand
0,515378344,2019-10-08,0,0.0,NaN,1106.819946,3,samsung
1,552415225,2019-10-10,0,0.0,NaN,54.029999,2,rowenta
2,542660200,2019-10-14,0,0.0,NaN,229.990005,4,samsung
3,516211454,2019-10-18,0,0.0,NaN,193.050003,1,dior


Рандомизация

In [8]:
import hashlib

def get_group(user_id):
    hash_object = hashlib.md5(str(user_id).encode())
    hash_value = int(hash_object.hexdigest(), 16)
    return 'A' if hash_value % 2 == 0 else 'B'

df['group'] = df['user_id'].apply(get_group)
df.head(4)

,user_id,event_date,is_purchased,total_revenue,avg_check,max_viewed_price,events_count,favorite_brand,group
0,515378344,2019-10-08,0,0.0,NaN,1106.819946,3,samsung,A
1,552415225,2019-10-10,0,0.0,NaN,54.029999,2,rowenta,A
2,542660200,2019-10-14,0,0.0,NaN,229.990005,4,samsung,A
3,516211454,2019-10-18,0,0.0,NaN,193.050003,1,dior,B


In [9]:
print(df['group'].value_counts())

print("\n Конверсия по группам:")
print(df.groupby('group')['is_purchased'].mean())

group
B    941359
A    941209
Name: count, dtype: int64

 Конверсия по группам:
group
A    0.106393
B    0.106648
Name: is_purchased, dtype: float64


Определяем размер выборки

In [10]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

base_conversion = df[df['group'] == 'A']['is_purchased'].mean()
expected_conv = df[df['group'] == 'A']['is_purchased'].mean() * 1.2

effect_size = proportion_effectsize(base_conversion, expected_conv)

power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(effect_size, power=0.8, alpha=0.05, ratio=1.0)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3575


Генериурем "синтетический" эффект в группе B

In [ ]:
import numpy as np

mde = 0.20 
baseline_cr = df[df['group'] == 'A']['is_purchased'].mean()

lift_prob = (baseline_cr * mde) / (1 - baseline_cr)

df['is_purchased_final'] = df['is_purchased'].copy()

mask_b_no_buy = (df['group'] == 'B') & (df['is_purchased'] == 0)

df.loc[mask_b_no_buy, 'is_purchased_final'] = np.random.binomial(
    n=1, 
    p=lift_prob, 
    size=mask_b_no_buy.sum()
)

print(f"Группа A CR: {df[df['group'] == 'A']['is_purchased_final'].mean():.4%}")
print(f"Группа B CR (синтетика): {df[df['group'] == 'B']['is_purchased_final'].mean():.4%}")

Группа A CR: 10.6393%
Группа B CR (синтетика): 12.8107%


In [12]:
from statsmodels.stats.proportion import proportions_ztest


z_stat, p_value = proportions_ztest(count = df.groupby('group')['is_purchased_final'].sum(), 
                                    nobs = df.groupby('group')['is_purchased_final'].count())

print(f"Z-статистика: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Результат статистически значим. Отклоняем нулевую гипотезу.")
else:
    print("Результат не статистически значим. Не отклоняем нулевую гипотезу.")

Z-статистика: -46.3037
P-value: 0.0000
Результат статистически значим. Отклоняем нулевую гипотезу.


In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

res = df.groupby('group')['is_purchased_final'].agg(['sum', 'count'])
count_a, nobs_a = res.loc['A', 'sum'], res.loc['A', 'count']
count_b, nobs_b = res.loc['B', 'sum'], res.loc['B', 'count']

ci_low, ci_upp = confint_proportions_2indep(
    count1=count_b, nobs1=nobs_b, 
    count2=count_a, nobs2=nobs_a, 
    method='wald'
)

diff = (count_b / nobs_b) - (count_a / nobs_a)

print(f"Абсолютная разница конверсий: {diff:.4%}")
print(f"95% CI для разности: [{ci_low:.4%}, {ci_upp:.4%}]")

Абсолютная разница конверсий: 2.1714%
95% CI для разности: [2.0796%, 2.2633%]


Перенос данных в Clickhouse

In [14]:
df

,user_id,event_date,is_purchased,total_revenue,avg_check,max_viewed_price,events_count,favorite_brand,group,is_purchased_final
0,515378344,2019-10-08,0,0.0,NaN,1106.819946,3,samsung,A,0
1,552415225,2019-10-10,0,0.0,NaN,54.029999,2,rowenta,A,0
2,542660200,2019-10-14,0,0.0,NaN,229.990005,4,samsung,A,0
3,516211454,2019-10-18,0,0.0,NaN,193.050003,1,dior,B,0
4,543794551,2019-10-08,0,0.0,NaN,102.709999,13,unknown,A,0
...,...,...,...,...,...,...,...,...,...,...
1882563,552598270,2019-10-07,0,0.0,NaN,486.079987,2,hubert,B,0
1882564,561551672,2019-10-18,0,0.0,NaN,249.860001,3,samsung,A,0
1882565,534282395,2019-10-16,0,0.0,NaN,34.380001,1,unknown,A,0
1882566,556789084,2019-10-21,0,0.0,NaN,74.650002,2,intex,A,0


In [16]:
df['avg_check'] = df['avg_check'].fillna(0)
df['max_viewed_price'] = df['max_viewed_price'].fillna(0)
df['event_date'] = pd.to_datetime(df['event_date'])

In [24]:
connection.command("""
CREATE TABLE IF NOT EXISTS ab_results (
    user_id UInt64,
    event_date DateTime,
    is_purchased UInt8,
    total_revenue Float64,
    avg_check Float64,
    max_viewed_price Float64,
    events_count UInt64,
    favorite_brand String,
    group String,
    is_purchased_final UInt8
) ENGINE = MergeTree()
ORDER BY (event_date, user_id)
""")

In [25]:
connection.insert_df(table='ab_results', df=df)

print(len(df))

1882568
